In [1]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 512
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.8.0+cu129
True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNeXtBlock(nn.Module):
    """
    ConvNeXt-style block with:
      - depthwise conv (5x5)
      - LayerNorm (over channels)
      - pointwise conv (expand 4x)
      - GELU
      - pointwise conv (project back)
      - optional layer scale
    """
    def __init__(self, dim, layer_scale_init_value=1e-3):
        super().__init__()
        # depthwise conv
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=5, padding=2, groups=dim)
        
        # LayerNorm over channels -> we need to permute to (N, H, W, C)
        self.norm = nn.LayerNorm(dim)

        # pointwise convs (implemented as 1x1 conv but on channels-last)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.pwconv2 = nn.Linear(4 * dim, dim)

        # layer scale
        if layer_scale_init_value > 0:
            self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim))
        else:
            self.gamma = None

    def forward(self, x):
        # x: (N, C, H, W)
        shortcut = x

        x = self.dwconv(x)              # (N, C, H, W)
        x = x.permute(0, 2, 3, 1)       # (N, H, W, C)
        x = self.norm(x)

        x = self.pwconv1(x)             # (N, H, W, 4C)
        x = F.gelu(x)
        x = self.pwconv2(x)             # (N, H, W, C)

        if self.gamma is not None:
            x = self.gamma * x

        x = x.permute(0, 3, 1, 2)       # (N, C, H, W)
        x = x + shortcut
        return x

class CrossStageGating(nn.Module):
    """
    Use a lower-stage feature (x_low) to gate a higher-stage feature (x_high).
    x_low:  (N, C_low, H_low, W_low)
    x_high: (N, C_high, H_high, W_high)
    """
    def __init__(self, c_low, c_high, reduction=4):
        super().__init__()
        hidden = max(c_low // reduction, 8)
        self.fc1 = nn.Linear(c_low, hidden)
        self.fc2 = nn.Linear(hidden, c_high)

    def forward(self, x_low, x_high):
        # global avg pool on low stage
        # x_low: (N, C_low, H_low, W_low)
        gap = x_low.mean(dim=[2, 3])        # (N, C_low)
        w = self.fc1(gap)                   # (N, hidden)
        w = F.gelu(w)
        w = self.fc2(w)                     # (N, C_high)
        w = torch.sigmoid(w).unsqueeze(-1).unsqueeze(-1)   # (N, C_high, 1, 1)
        return x_high * w                   # gated high-stage feature

class DownsampleLayer(nn.Module):
    """
    Simple downsampling: 2x2 conv with stride 2.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)

    def forward(self, x):
        return self.conv(x)


class ConvNeXtStudent64(nn.Module):
    """
    ConvNeXt-style model optimized for 64x64 images and <=15 epochs.
    
    Stages:
      - Stem: 4x4 conv, stride 4 -> 64x64 -> 16x16
      - Stage1: [2 blocks], C=64
      - Downsample -> 32x32? NO (we already did stride4); we go 16->8->4
      - Stage2: [2 blocks], C=128
      - Stage3: [4 blocks], C=256, with cross-stage gating using Stage2.
    """
    def __init__(self, in_ch=3, num_classes=15, layer_scale_init_value=1e-3):
        super().__init__()
        # stem: 64x64 -> 16x16, channels=64
        self.stem = nn.Conv2d(in_ch, 128, kernel_size=4, stride=4)

        # stage 1 (16x16)
        self.stage1 = nn.Sequential(
            ConvNeXtBlock(128, layer_scale_init_value),
            ConvNeXtBlock(128, layer_scale_init_value),
        )

        # downsample 1: 16x16 -> 8x8
        self.down1 = DownsampleLayer(128, 256)

        # stage 2 (8x8)
        self.stage2 = nn.Sequential(
            ConvNeXtBlock(256, layer_scale_init_value),
            ConvNeXtBlock(256, layer_scale_init_value),
        )

        # downsample 2: 8x8 -> 4x4
        self.down2 = DownsampleLayer(256, 512)

        # stage 3 (4x4)
        self.stage3 = nn.Sequential(
            ConvNeXtBlock(512, layer_scale_init_value),
            ConvNeXtBlock(512, layer_scale_init_value),
            ConvNeXtBlock(512, layer_scale_init_value),
            ConvNeXtBlock(512, layer_scale_init_value),
        )

        # cross-stage gating: use stage2 to gate stage3
        self.gating = CrossStageGating(c_low=256, c_high=512, reduction=4)

        # classifier head
        self.norm = nn.LayerNorm(512)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # x: (N, 3, 64, 64)
        x = self.stem(x)          # (N, 64, 16, 16)
        x = self.stage1(x)        # (N, 64, 16, 16)

        x2 = self.down1(x)        # (N, 128, 8, 8)
        x2 = self.stage2(x2)      # (N, 128, 8, 8)

        x3 = self.down2(x2)       # (N, 256, 4, 4)
        x3 = self.stage3(x3)      # (N, 256, 4, 4)

        # cross-stage gating
        # x3 = self.gating(x2, x3)  # stage2 info -> gate stage3

        # global average pooling
        x3 = x3.mean(dim=[2, 3])  # (N, 256)

        # LayerNorm then FC
        x3 = self.norm(x3)
        out = self.fc(x3)         # (N, num_classes)
        return out

In [6]:
model = ConvNeXtStudent64(in_ch=3, num_classes=15, layer_scale_init_value=0.0).to(device)

# Model parameter checking

In [7]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 10513103
Parameter usage : 10.513103%


# Model training

In [8]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [9]:
import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast

# 0. 벤치마크 켜기 (속도 향상)
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

epochs = 15
save_path = "best_model.pth"
best_val_loss = float("inf")

criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # 통계 (AMP로 계산된 output을 그대로 사용)
        train_loss_sum += loss.item() * y.size(0)
        
        # 예측값 계산 (여기는 그라디언트 필요 없으므로 detach 추천)
        preds = torch.argmax(output.detach(), dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # VALIDATION (그대로 유지)
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # 검증 때는 AMP를 굳이 안 써도 되지만, 쓰면 조금 더 빠를 수 있음
            with autocast():
                output = model(x)
                val_loss_batch = criterion(output, y)

            val_loss_sum += val_loss_batch.item() * y.size(0)
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    scheduler.step()

    # MODEL SAVE
    save_dict = {
        "epoch": epoch,
        "model_state_dict": (
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    torch.save(save_dict, f'epoch_{epoch}.pth')

    # BEST MODEL SPECIFICATION
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        save_dict = {
            "epoch": epoch,
            "model_state_dict": (
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(save_dict, save_path)
        print(f"Best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

Epoch 0 [Val]:   0%|          | 0/18 [00:00<?, ?it/s]C:\Users\MAIN\AppData\Local\Temp\ipykernel_18860\3605441430.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 18/18 [00:12<00:00,  1.41it/s]


Epoch 00 | Train Loss: 2.7474 | Train Acc: 0.1111 | Val Loss: 2.5617 | Val Acc: 0.1568
Best model saved at epoch 0 (val_loss=2.5617)


Epoch 1 [Val]: 100%|██████████| 18/18 [00:12<00:00,  1.46it/s]


Epoch 01 | Train Loss: 2.4506 | Train Acc: 0.1918 | Val Loss: 2.3822 | Val Acc: 0.2251
Best model saved at epoch 1 (val_loss=2.3822)


Epoch 2 [Val]: 100%|██████████| 18/18 [00:12<00:00,  1.46it/s]


...

Epoch 12 | Train Loss: 1.5890 | Train Acc: 0.4863 | Val Loss: 1.6622 | Val Acc: 0.4634
Best model saved at epoch 12 (val_loss=1.6622)


Epoch 13 [Val]: 100%|██████████| 18/18 [00:12<00:00,  1.45it/s]


Epoch 13 | Train Loss: 1.5641 | Train Acc: 0.4927 | Val Loss: 1.6649 | Val Acc: 0.4666


Epoch 14 [Val]: 100%|██████████| 18/18 [00:12<00:00,  1.45it/s]

Epoch 14 | Train Loss: 1.5547 | Train Acc: 0.4965 | Val Loss: 1.6580 | Val Acc: 0.4658
Best model saved at epoch 14 (val_loss=1.6580)


In [10]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

'\n# 모델 불러오기\ncheckpoint = torch.load("best_model.pth", map_location=device)\n\n# 먼저 순수 모델을 만들고 로드\nbase_model = ConvNeXtBN(num_classes=15)\nbase_model.load_state_dict(checkpoint["model_state_dict"])\n\n# 그 다음에 DataParallel로 감쌈\nif torch.cuda.device_count() > 1:\n    model = torch.nn.DataParallel(base_model)\nelse:\n    model = base_model\n\nmodel = model.to(device)\n\noptimizer = torch.optim.Adam(model.parameters(), lr=0.001)\noptimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n'

In [11]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

'\n# 모델 로드 검증\nmissing_keys, unexpected_keys = base_model.load_state_dict(\n    checkpoint["model_state_dict"], strict=False\n)\n\nprint("Missing keys:", missing_keys)\nprint("Unexpected keys:", unexpected_keys)\n\n# 파라미터 값 확인\nwith torch.no_grad():\n    w = base_model.downsample_layers[0][0].weight\n\nprint("Sample weight stats:")\nprint("  mean:", w.mean().item())\nprint("  std :", w.std().item())\nprint("  min :", w.min().item())\nprint("  max :", w.max().item())\n\n# forward 테스트\nmodel.eval()\nwith torch.no_grad():\n    dummy = torch.randn(2, 3, 224, 224).to(device)\n    out = model(dummy)\n\nprint("Output shape:", out.shape)\nprint("Has NaN:", torch.isnan(out).any().item())\n'

In [12]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

'\nprint("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))\nprint("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))\nprint("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))\n'

# Submit
Do not edit the submission code below.

In [13]:
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 10513103
Parameter usage : 10.513103%


100%|██████████| 15/15 [00:12<00:00,  1.21it/s]
